# run_simulation.ipynb

Main simulation runner. Two modes:

- **Section 5A — Full matrix run**: runs all configs in `RUN_MATRIX` back-to-back, logs cost + latency per run. Use this for the full evaluation.
- **Section 5B — Single-run step-through**: run one config interactively, day-by-day. Use this for development and inspection.

**Run matrix (Section 2 defines this):**

| Run label | Model | Judge | Ablation |
|---|---|---|---|
| `Baseline_Full` | claude-sonnet-4-6 | claude-opus-4-6 | Full (memory + retrieval + reflection) |
| `Baseline_No_Reflection` | claude-sonnet-4-6 | claude-opus-4-6 | Memory + retrieval, no reflection |
| `Baseline_No_Memory_No_Retrieval_No_Reflection` | claude-sonnet-4-6 | claude-opus-4-6 | Seed personality only |
| `Budget_Full` | claude-haiku-4-5 | claude-opus-4-6 | Full (memory + retrieval + reflection) |

Decision and reflection use the same model within each run. To change models, update `MODEL_BASELINE` or `MODEL_BUDGET` in Section 2.

After running, hand the `outputs/runs/` JSONLs to `run_evaluation.ipynb` for judging and Excel export.

**Ablation variants (Experiment 1):**
- Variant 1 — Full system: memory stream + retrieval + reflection
- Variant 2 — No reflection: memory stream + retrieval, no reflection
- Variant 3 — Seed personality only: no memory stream, no retrieval, no reflection

---
## Section 1 — Setup

Install packages and initialise API clients. Run once per Colab session.

In [ ]:
!pip install -q --disable-pip-version-check --no-warn-script-location \
  anthropic openai \
  sentence-transformers \
  'numpy>=1.26' \
  'pandas>=2.2' \
  scipy \
  scikit-learn \
  pyyaml

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_PATH = '/content/drive/MyDrive/Spring2026/fgenai/SIMULATION/berkeley-homes-wildfire-agent-simulation'
except ImportError:
    PROJECT_PATH = os.path.abspath('..')   # running locally: one level up from notebooks/

os.chdir(PROJECT_PATH)
print(f'Working directory: {os.getcwd()}')

In [ ]:
import sys
import json
import yaml
import time
import pandas as pd
from pathlib import Path
from datetime import datetime

sys.path.insert(0, PROJECT_PATH)

from src.engine.simulation import Simulation, SimulationConfig
from src.llm.client import Config, init_clients, init_openrouter_client, usage_tracker
from src.agents.retrieval import RetrievalConfig
from src.agents.reflection import ReflectionConfig

print('Imports OK')

In [ ]:
# Always initialise Anthropic. OpenRouter required for Kimi and other non-Claude models.
client_anthropic  = init_clients()
client_openrouter = init_openrouter_client()

---
## Section 2 — Run matrix

**This is the only cell you need to change between experiments.**

- Comment out any run configs you don't want to execute this session.
- `CONCISE_OUTPUT=True` keeps decision/reasoning to 1-2 sentences — recommended for all evaluation runs.
- `AGENT_PATHS` and `SCENARIO_PATH` are shared across all runs.
- Retrieval and reflection hyperparameters are locked from Stage 2 validation.

In [ ]:
SCENARIO_PATH = 'config/scenarios/baseline.yaml'

AGENT_PATHS = [
    'config/agents/selected/jennifer.yaml',
    'config/agents/selected/beth.yaml',
    'config/agents/selected/edward.yaml',
    'config/agents/selected/lola.yaml',
    'config/agents/selected/synthetic_non_compliant.yaml',
]

# Locked from Stage 2 validation (stage2_validation_beth_v2.ipynb)
RETRIEVAL_CFG  = RetrievalConfig(top_k=8, recency_weight=1.0, importance_weight=1.0, relevance_weight=2.0)
REFLECTION_CFG = ReflectionConfig(threshold=50.0, num_questions=3)

# ── Model selection ────────────────────────────────────────────────────────────
# Decision and reflection use the same model within each run.
MODEL_BASELINE = 'claude-sonnet-4-6'
MODEL_BUDGET   = 'claude-haiku-4-5-20251001'
MODEL_JUDGE    = 'claude-opus-4-6'

# ── Run matrix ─────────────────────────────────────────────────────────────────
# Baseline_Full is first — run that alone to verify the pipeline before the others.
# When happy, set RUNS_TO_EXECUTE = None in Section 5A to run everything.

RUN_MATRIX = [

    # 1. Baseline: full system (Variant 1) — test this first
    SimulationConfig(
        run_label        = 'Baseline_Full',
        scenario_path    = SCENARIO_PATH,
        agent_yaml_paths = AGENT_PATHS,
        llm_config       = Config(
            DECISION_MODEL   = MODEL_BASELINE,
            REFLECTION_MODEL = MODEL_BASELINE,
            JUDGE_MODEL      = MODEL_JUDGE,
            CONCISE_OUTPUT   = True,
        ),
        retrieval_config  = RETRIEVAL_CFG,
        reflection_config = REFLECTION_CFG,
        use_memory        = True,
        use_reflection    = True,
    ),

    # 2. Baseline ablation: no reflection (Variant 2)
    SimulationConfig(
        run_label        = 'Baseline_No_Reflection',
        scenario_path    = SCENARIO_PATH,
        agent_yaml_paths = AGENT_PATHS,
        llm_config       = Config(
            DECISION_MODEL   = MODEL_BASELINE,
            REFLECTION_MODEL = MODEL_BASELINE,
            JUDGE_MODEL      = MODEL_JUDGE,
            CONCISE_OUTPUT   = True,
        ),
        retrieval_config  = RETRIEVAL_CFG,
        reflection_config = REFLECTION_CFG,
        use_memory        = True,
        use_reflection    = False,   # Variant 2: memory + retrieval, no reflection
    ),

    # 3. Baseline ablation: seed personality only (Variant 3)
    SimulationConfig(
        run_label        = 'Baseline_No_Memory_No_Retrieval_No_Reflection',
        scenario_path    = SCENARIO_PATH,
        agent_yaml_paths = AGENT_PATHS,
        llm_config       = Config(
            DECISION_MODEL   = MODEL_BASELINE,
            REFLECTION_MODEL = MODEL_BASELINE,
            JUDGE_MODEL      = MODEL_JUDGE,
            CONCISE_OUTPUT   = True,
        ),
        retrieval_config  = RETRIEVAL_CFG,
        reflection_config = REFLECTION_CFG,
        use_memory        = False,   # Variant 3: no memory stream, retrieval, or reflection
        use_reflection    = False,
    ),

    # 4. Budget: Haiku, full system — cost comparison
    SimulationConfig(
        run_label        = 'Budget_Full',
        scenario_path    = SCENARIO_PATH,
        agent_yaml_paths = AGENT_PATHS,
        llm_config       = Config(
            DECISION_MODEL   = MODEL_BUDGET,
            REFLECTION_MODEL = MODEL_BUDGET,
            JUDGE_MODEL      = MODEL_JUDGE,
            CONCISE_OUTPUT   = True,
        ),
        retrieval_config  = RETRIEVAL_CFG,
        reflection_config = REFLECTION_CFG,
        use_memory        = True,
        use_reflection    = True,
    ),

]

print(f'{len(RUN_MATRIX)} runs configured:')
for rc in RUN_MATRIX:
    print(f'\n  {rc.run_label}')
    print(rc.describe())

---
## Section 3 — Preview

Inspect the event timeline and agent profiles **before running anything**. No LLM calls.

In [ ]:
with open(SCENARIO_PATH) as f:
    scenario = yaml.safe_load(f)

print(f"Scenario: {scenario['scenario_name']}")
print(f"Duration: {scenario['duration_days']} days")
print(f"Description: {scenario.get('description', '')}\n")

rows = [
    {
        'Day': e['day'],
        'Type': e['type'],
        'Channel': e['channel'],
        'Targets': str(e.get('target_agents', 'all'))[:40],
        'Content preview': str(e.get('content', ''))[:80] + '...',
    }
    for e in scenario['events']
]
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
for path in AGENT_PATHS:
    with open(path) as f:
        raw = yaml.safe_load(f)
    data = raw['agents'][0] if 'agents' in raw else raw

    print(f"\n{'='*60}")
    print(f"  {data['display_name']}  (id: {data['id']})")
    print(f"  Compliance: {data.get('compliance_status', '?')}  "
          f"Insurance: {data.get('insurance_status', '?')}")
    print(f"  Memory seeds: {len(data.get('memory_seeds', []))}")
    print(f"  Seed narrative: {data['seed_narrative'][:120].strip()}...")

---
## Section 5A — Full matrix run

Runs every config in `RUN_MATRIX` back-to-back.
Saves one JSONL per run to `outputs/runs/` with cost + latency as the final entry.

To run a subset, set `RUNS_TO_EXECUTE` to a list of labels, e.g. `['Premium_Full']`.

**Expected time:** ~5-20 min per run depending on model and number of agents.

Hand the output JSONLs to `run_evaluation.ipynb` when done.

In [ ]:
# ── Which runs to execute this session ────────────────────────────────────────
# Start with Baseline_Full to verify the pipeline end-to-end.
# Once happy, set to None to run the full matrix.
RUNS_TO_EXECUTE = ['Baseline_Full']
# RUNS_TO_EXECUTE = None  # uncomment to run all four

run_summaries = []

for sim_config in RUN_MATRIX:
    if RUNS_TO_EXECUTE and sim_config.run_label not in RUNS_TO_EXECUTE:
        print(f'SKIP {sim_config.run_label}')
        continue

    print(f'\n{"#"*60}')
    print(f'  Starting: {sim_config.run_label}')
    print(f'{"#"*60}')
    print(sim_config.describe())

    usage_tracker.reset()
    run_start = time.time()

    sim = Simulation(
        sim_config        = sim_config,
        client_anthropic  = client_anthropic,
        client_openrouter = client_openrouter,
    )
    sim.run(verbose=True)

    latency   = time.time() - run_start
    cost_info = usage_tracker.to_dict(
        agent_model = sim_config.llm_config.DECISION_MODEL,
        judge_model = sim_config.llm_config.JUDGE_MODEL,
    )

    sim.logger.log_run_summary(latency_seconds=latency, cost_info=cost_info)
    sim.logger.close()

    summary = {
        'run_label':       sim_config.run_label,
        'run_id':          sim_config.run_id,
        'latency_s':       round(latency, 1),
        'agent_cost_usd':  cost_info['agent_cost_usd'],
        'total_cost_usd':  cost_info['total_cost_usd'],
        'jsonl':           f"outputs/runs/{sim_config.run_id}.jsonl",
    }
    run_summaries.append(summary)
    print(f'\n  Done: {sim_config.run_label} | {latency:.1f}s | '
          f'agent=${cost_info["agent_cost_usd"]:.4f} | total=${cost_info["total_cost_usd"]:.4f}')

print(f'\n{"="*60}')
print('DONE')
print('='*60)
if run_summaries:
    print(pd.DataFrame(run_summaries)[
        ['run_label', 'latency_s', 'agent_cost_usd', 'total_cost_usd', 'jsonl']
    ].to_string(index=False))
print('\nNext: open run_evaluation.ipynb → Section 3 will pick up these JSONLs automatically.')

---
## Section 5B — Single-run step-through

Use for development and inspection. Pick one config from `RUN_MATRIX` by index and
step through it day-by-day. **Only run one of 5A or 5B per session.**

In [ ]:
# Pick a config by RUN_MATRIX index (0=Premium_Full, 1=Baseline_Full, etc.)
SINGLE_RUN_INDEX = 0

sim_config = RUN_MATRIX[SINGLE_RUN_INDEX]
print(sim_config.describe())

usage_tracker.reset()
_single_run_start = time.time()

sim = Simulation(
    sim_config        = sim_config,
    client_anthropic  = client_anthropic,
    client_openrouter = client_openrouter,
)
print('\nReady. Re-run the step cell below to advance one day at a time.')

In [ ]:
# Re-run this cell to advance one simulation day
tick_result = sim.step()

if tick_result is None:
    latency   = time.time() - _single_run_start
    cost_info = usage_tracker.to_dict(
        agent_model = sim_config.llm_config.DECISION_MODEL,
        judge_model = sim_config.llm_config.JUDGE_MODEL,
    )
    sim.logger.log_run_summary(latency_seconds=latency, cost_info=cost_info)
    sim.close()
    print(f'Simulation complete. {latency:.1f}s | total cost ${cost_info["total_cost_usd"]:.4f}')
    print(f'JSONL: outputs/runs/{sim_config.run_id}.jsonl')
elif tick_result['events_processed']:
    tick = tick_result['tick']
    for er in tick_result['events_processed']:
        print(f"\nDay {tick} | {er['event_type'].upper()} | {er['channel']}")
        print(f"  → {er['content'][:100]}...")
        for resp in er['agent_responses']:
            print(f"\n  [{resp['agent_display_name']}]")
            print(f"    DECISION:  {resp['decision'][:120]}")
            print(f"    REASONING: {resp['reasoning'][:120]}")
            if resp['new_reflections']:
                print(f"    ★ {len(resp['new_reflections'])} reflection(s) fired")
else:
    print(f"Day {tick_result['tick']} — no events scheduled")

---
## Section 6 — Inspect (single-run mode)

In [ ]:
def inspect_agent(display_name: str):
    agent = sim.agents.get(display_name)
    if agent is None:
        print(f"Not found. Available: {list(sim.agents.keys())}")
        return
    agent.pretty_print()


def inspect_memory(display_name: str, n: int = None, memory_type: str = None):
    """Print memory stream. memory_type: 'observation'|'decision'|'reflection'|'conversation'"""
    agent = sim.agents.get(display_name)
    if agent is None:
        print(f"Not found. Available: {list(sim.agents.keys())}")
        return
    memories = agent.memory.get_all()
    if memory_type:
        memories = [m for m in memories if m.type == memory_type]
    if n:
        memories = memories[-n:]
    print(f"{display_name} — {len(memories)} memories")
    for m in memories:
        print(f"  [{m.id}] Day {m.timestamp} | {m.type} | imp={m.importance} | {m.description[:100]}")


def compare_agents(event_type: str = None):
    for tick_result in sim.tick_results:
        for er in tick_result['events_processed']:
            if event_type and er['event_type'] != event_type:
                continue
            print(f"\nDay {tick_result['tick']} | {er['event_type'].upper()}")
            for resp in er['agent_responses']:
                print(f"  [{resp['agent_display_name']}] {resp['decision'][:120]}")


def show_all_reflections():
    found = False
    for tick_result in sim.tick_results:
        for er in tick_result['events_processed']:
            for resp in er['agent_responses']:
                if resp['new_reflections']:
                    found = True
                    print(f"\nDay {tick_result['tick']} | {resp['agent_display_name']}")
                    for r in resp['new_reflections']:
                        print(f"  ★ {r.description[:120]}")
    if not found:
        print('No reflections fired yet.')


print('Helpers defined: inspect_agent(), inspect_memory(), compare_agents(), show_all_reflections()')

---
## Note: held_out_responses and simulation evaluation

Agent YAMLs contain `held_out_responses` — real interview quotes tied to specific scenarios.
**These are not used in the main simulation evaluation.**

By the time a matching event fires, the agent has accumulated new memories and may have
reflected — divergence from the interview quote is expected and intentional.
Held-outs are Stage 2 pre-simulation validation only.

See `System Design and Notes/evaluation_plan.md` for full rationale and `System Design and Notes/systemdesign.md` for the data split.